# TOMAS ARC-AGI-3 Solver v2.4.1 — GPU-Enabled Kaggle Notebook

Taiyi Mutual-Play (TOMAS) framework for ARC-AGI-3 video reasoning.

**⚡ GPU Acceleration Enabled**  
- Numba JIT CUDA kernels (7 @cuda.jit kernels)  
- Config: `cuda.enabled = true`, `use_numba = true`  
- T4 GPU (16GB VRAM) fully utilized

**v2.4 Features:**  
- ψ-Gate Semantic Gating (enabled)  
- AEGIS Evolution Engine (enabled)  
- Causal DSL Prior (enabled)  
- RHAE Evaluation Framework  
- Numba JIT + CUDA GPU acceleration

## 1. Environment Setup (GPU Check + Dependencies)

In [ ]:
import sys
import os

# === Install dependencies (CUDA-enabled) ===
print('=== Installing Dependencies ===')
!pip install -q numpy scipy networkx pyyaml loguru tqdm httpx
!pip install -q numba  # CUDA JIT support

# === GPU Availability Check ===
print('\n=== GPU Detection ===')

# Check 1: PyTorch CUDA
try:
    import torch
    print(f'✅ PyTorch {torch.__version__}')
    print(f'   CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        gpu_id = 0
        print(f'   GPU: {torch.cuda.get_device_name(gpu_id)}')
        vram_gb = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
        print(f'   VRAM: {vram_gb:.1f} GB')
        # Quick GPU test
        x = torch.rand(1000, 1000, device='cuda')
        y = torch.mm(x, x)
        print(f'   GPU test: OK (matmul 1000x1000)')
except Exception as e:
    print(f'⚠️  PyTorch CUDA check failed: {e}')

# Check 2: Numba CUDA
try:
    from numba import cuda, njit
    print(f'\n✅ Numba {__import__("numba").__version__}')
    print(f'   CUDA available: {cuda.is_available()}')
    if cuda.is_available():
        print(f'   CUDA devices: {cuda.get_num_devices()}')
        @cuda.jit
        def test_kernel(x, y):
            i = cuda.grid(1)
            if i < x.shape[0]:
                y[i] = x[i] * 2
        # Quick test
        import numpy as np
        x_np = np.random.rand(1000).astype(np.float32)
        y_np = np.zeros_like(x_np)
        test_kernel[1, 256](x_np, y_np)
        print(f'   Numba CUDA kernel test: OK')
except Exception as e:
    print(f'\n⚠️  Numba CUDA check failed: {e}')
    print('   → TOMAS will use CPU fallback (slower)')

print('\n=== Environment Ready ===')

## 2. Load TOMAS Solver v2.4 (with CUDA)

In [ ]:
import sys
import json
from pathlib import Path

# Add solver to path
solver_path = '/kaggle/input/tomas-arc3-solver'
if solver_path not in sys.path:
    sys.path.insert(0, solver_path)

from src.utils.config import load_config
from src.solver.tomas_solver import TOMASSolver

# Load config
config_path = f'{solver_path}/config/default.yaml'
config = load_config(config_path)

# === Kaggle Environment Overrides ===
config['kaggle'] = {
    'input_dir': '/kaggle/input/arc-agi-3',
    'output_dir': '/kaggle/working',
}
config['vl_api'] = {'available': False}  # No external API

# === CUDA GPU Configuration (CRITICAL) ===
if 'cuda' not in config:
    config['cuda'] = {}
config['cuda']['enabled'] = True
config['cuda']['use_numba'] = True
config['cuda']['max_vram_gb'] = 16  # T4 GPU
config['cuda']['block_size'] = 256
config['cuda']['grid_size'] = 64

# === v2.4 Feature Configuration ===
if 'psi_gate' not in config:
    config['psi_gate'] = {}
config['psi_gate']['enabled'] = True
config['psi_gate']['use_default_anchors'] = True
config['psi_gate']['tolerance_decay_rate'] = 0.05

if 'aegis' not in config:
    config['aegis'] = {}
config['aegis']['enabled'] = True
config['aegis']['max_generations'] = 3
config['aegis']['population_size'] = 20
config['aegis']['mutation_rate'] = 0.15

if 'causal_prior' not in config:
    config['causal_prior'] = {}
config['causal_prior']['enabled'] = True

if 'eval' not in config:
    config['eval'] = {}
config['eval']['enabled'] = False  # Disable in competition

print('=== TOMAS v2.4 Configuration ===')
print(f'  CUDA enabled: {config["cuda"]["enabled"]}')
print(f'  CUDA use_numba: {config["cuda"]["use_numba"]}')
print(f'  Max VRAM: {config["cuda"]["max_vram_gb"]} GB')
print(f'  ψ-Gate: {"ON" if config["psi_gate"]["enabled"] else "OFF"}')
print(f'  AEGIS: {"ON" if config["aegis"]["enabled"] else "OFF"}')
print(f'  Causal Prior: {"ON" if config["causal_prior"]["enabled"] else "OFF"}')

## 3. Initialize Solver (Verify CUDA Backend)

In [ ]:
print('=== Initializing TOMAS Solver ===')
solver = TOMASSolver(config)

print(f'\n✅ Solver initialized')
print(f'  Mode: {solver.mode}')
print(f'  Search max_depth: {solver.searcher.max_depth}')
print(f'  Search mdl_threshold: {solver.searcher.mdl_threshold}')

# Check CUDA backend
if hasattr(solver.searcher, '_cuda_backend'):
    print(f'\n✅ CUDA Backend: {solver.searcher._cuda_backend}')
else:
    print(f'\n⚠️  CUDA Backend: Not detected (will use CPU)')

# Check psi-gate
if hasattr(solver, 'psi_gate') and solver.psi_gate:
    print(f'\n✅ ψ-Gate: Active ({len(solver.psi_gate.anchors)} anchors)')
    print(f'   Multi-world: {solver.psi_gate.enable_multi_world}')
else:
    print(f'\n⚠️  ψ-Gate: Disabled')

# Check pruning optimizer
if hasattr(solver.searcher, 'pruning') and solver.searcher.pruning:
    print(f'\n✅ Pruning Optimizer: Active')
    if hasattr(solver.searcher.pruning, 'strategies'):
        print(f'   Strategies: {len(solver.searcher.pruning.strategies)}')
else:
    print(f'\n⚠️  Pruning Optimizer: Disabled')

## 4. Load Competition Data

In [ ]:
from pathlib import Path
import json

input_dir = Path('/kaggle/input/arc-agi-3')
task_files = sorted(input_dir.glob('**/*.json'))
print(f'=== Competition Data ===')
print(f'  Found {len(task_files)} task files')

# Preview first task
if task_files:
    with open(task_files[0], 'r') as f:
        first_task = json.load(f)
    print(f'\nSample task: {task_files[0].name}')
    print(f'  Train pairs: {len(first_task.get("train", []))}')
    print(f'  Test pairs: {len(first_task.get("test", []))}')

# Also check evaluation (if available)
eval_dir = Path('/kaggle/input/arc-agi-3-evaluation')
if eval_dir.exists():
    eval_files = sorted(eval_dir.glob('**/*.json'))
    print(f'\nEvaluation data: {len(eval_files)} tasks')
    config['kaggle']['has_evaluation'] = True
else:
    config['kaggle']['has_evaluation'] = False
    print(f'\nNo evaluation data (submission-only mode)')

## 5. Solve All Tasks (with CUDA Acceleration)

In [ ]:
from tqdm import tqdm
import time
import numpy as np

results = {}
stats = {
    'total': 0, 'solved': 0, 'failed': 0,
    'total_time': 0, 'total_candidates': 0,
    'cuda_used': False,
}
start_time = time.time()

print('=== Solving All Tasks (CUDA Accelerated) ===')
print(f'  Tasks: {len(task_files)}')
print(f'  Time budget: 80 seconds')
print(f'  CUDA enabled: {config["cuda"]["enabled"]}')

for tf in tqdm(task_files, desc='Solving (CUDA)'):
    try:
        with open(tf, 'r') as f:
            task_data = json.load(f)
        task_data['task_id'] = tf.stem
        
        stats['total'] += 1
        task_start = time.time()
        
        # Auto-select mode based on remaining time
        remaining_time = max(10.0, 80.0 - (time.time() - start_time) / max(stats['total'], 1))
        mode = solver.auto_select_mode(remaining_time, len(task_data.get('train', [])))
        
        result = solver.solve(task_data, mode=mode)
        results[tf.stem] = result
        
        task_time = time.time() - task_start
        
        if result is not None:
            stats['solved'] += 1
            tqdm.write(f'  ✅ {tf.stem}: SOLVED ({task_time:.1f}s, mode={mode})')
        else:
            stats['failed'] += 1
            tqdm.write(f'  ❌ {tf.stem}: FAILED ({task_time:.1f}s, mode={mode})')
        
        # Library Learning feedback
        solver._post_solve_learning(result)
        
        # Check if CUDA was used
        if hasattr(solver.searcher, '_cuda_backend') and solver.searcher._cuda_backend:
            stats['cuda_used'] = True
        
    except Exception as e:
        print(f'\nError solving {tf.name}: {e}')
        results[tf.stem] = None
        stats['failed'] += 1

elapsed = time.time() - start_time
stats['total_time'] = elapsed

print(f'\n=== Solving Complete ===')
print(f'  Total: {stats["total"]}')
print(f'  Solved: {stats["solved"]} ({stats["solved"]/max(stats["total"],1)*100:.1f}%)')
print(f'  Failed: {stats["failed"]}')
print(f'  Time: {elapsed:.1f}s ({elapsed/max(stats["total"],1):.1f}s/task)')
print(f'  CUDA used: {stats["cuda_used"]}')

## 6. AEGIS Evolution (Retry Failed Tasks)

In [ ]:
# Re-solve failed tasks with AEGIS evolution engine
if config['aegis']['enabled'] and stats['failed'] > 0:
    print(f'\n=== AEGIS Evolution Retry ({stats["failed"]} failed tasks) ===')
    
    from src.solver.aegis_evolver import AEGISEvolver
    aegis = AEGISEvolver(config.get('aegis', {}))
    
    for task_id, result in list(results.items()):
        if result is None:
            print(f'\nRetrying {task_id} with AEGIS evolution...')
            try:
                task_file = next(tf for tf in task_files if tf.stem == task_id)
                with open(task_file, 'r') as f:
                    task_data = json.load(f)
                
                evolved_result = aegis.evolve_search(
                    task_data, solver.searcher, max_generations=3
                )
                if evolved_result:
                    results[task_id] = evolved_result
                    stats['solved'] += 1
                    stats['failed'] -= 1
                    print(f'  ✅ Solved via AEGIS!')
                else:
                    print(f'  ❌ Still failed')
            except Exception as e:
                print(f'  ❌ AEGIS error: {e}')
else:
    print('AEGIS disabled or no failed tasks to retry.')'

print(f'\n=== Final Stats ===')
print(f'  Solved: {stats["solved"]} / {stats["total"]} ({stats["solved"]/max(stats["total"],1)*100:.1f}%)')

## 7. Generate Submission (Kaggle Format)

In [ ]:
from src.utils.kaggle_format import KaggleFormatAdapter
import json

adapter = KaggleFormatAdapter()
submission = {}
for task_id, pred in results.items():
    if pred is not None:
        submission[task_id] = pred

output_path = '/kaggle/working/submission.json'
with open(output_path, 'w') as f:
    json.dump(submission, f)

print('=== Submission Generated ===')
print(f'  Path: {output_path}')
print(f'  Tasks submitted: {len(submission)} / {stats["total"]}')
print(f'  Submission rate: {len(submission)/max(stats["total"],1)*100:.1f}%')

# Validate output format
valid = adapter.validate_output(submission)
print(f'  Format valid: {valid}')

## 8. Verify Output + Final Summary

In [ ]:
import os
import time

print('=== Final Summary ===')
print(f'  Version: 2.4.1 (CUDA Enabled)')
print(f'  Total tasks: {stats["total"]}')
print(f'  Submitted: {len(submission)}')
print(f'  Success rate: {len(submission)/max(stats["total"],1)*100:.1f}%')
print(f'  Total time: {stats["total_time"]:.1f}s')
print(f'  Avg time/task: {stats["total_time"]/max(stats["total"],1):.1f}s')
print(f'  CUDA used: {stats["cuda_used"]}')
print(f'  ψ-Gate: {"ON" if config["psi_gate"]["enabled"] else "OFF"}')
print(f'  AEGIS: {"ON" if config["aegis"]["enabled"] else "OFF"}')
print(f'  Causal Prior: {"ON" if config["causal_prior"]["enabled"] else "OFF"}')

# Output file info
output_size = os.path.getsize(output_path) if os.path.exists(output_path) else 0
print(f'\nOutput file: {output_path}')
print(f'  Size: {output_size} bytes ({output_size/1024:.1f} KB)')

# Save run log
log_data = {
    'version': '2.4.1',
    'stats': stats,
    'config': {
        'cuda': config['cuda']['enabled'],
        'psi_gate': config['psi_gate']['enabled'],
        'aegis': config['aegis']['enabled'],
        'causal_prior': config['causal_prior']['enabled'],
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open('/kaggle/working/run_log.json', 'w') as f:
    json.dump(log_data, f, indent=2)
print(f'\nRun log saved to /kaggle/working/run_log.json')

## 9. Post-Submission Analysis (Optional)

In [ ]:
# Analyze pruning effectiveness
if hasattr(solver.searcher, 'pruning') and solver.searcher.pruning:
    pruning_stats = solver.searcher.pruning.stats
    print('=== Pruning Statistics ===')
    total_pruned = 0
    for strategy, count in pruning_stats.items():
        print(f'  {strategy}: {count}')
        if isinstance(count, (int, float)):
            total_pruned += count
    print(f'  Total pruned: {total_pruned}')

# Analyze solver performance
print(f'\n=== Performance ===')
print(f'  Avg time/task: {stats["total_time"]/max(stats["total"],1):.1f}s')
if stats['cuda_used']:
    print(f'  ✅ CUDA acceleration: Active')
else:
    print(f'  ⚠️  CUDA acceleration: Inactive (check Numba installation)')

# Show config summary
print(f'\n=== Configuration ===')
print(f'  CUDA enabled: {config["cuda"]["enabled"]}')
print(f'  CUDA use_numba: {config["cuda"]["use_numba"]}')
print(f'  ψ-Gate enabled: {config["psi_gate"]["enabled"]}')
print(f'  AEGIS enabled: {config["aegis"]["enabled"]}')